# Melampo: misura dell'interprete delle immagini (VoxTell + TotalSegmentator)

Solo dati pubblici: Medical Segmentation Decathlon, Task06 Lung (tumori polmonari in TC, licenza CC BY-SA 4.0).

Prima di eseguire, nelle impostazioni del notebook (pannello a destra):
- **Accelerator**: GPU T4 x2 (oppure P100);
- **Internet**: on (serve la verifica del telefono sull'account Kaggle);
- se il repository è privato: **Add-ons → Secrets → GITHUB_TOKEN** con un token GitHub *fine-grained*, solo lettura, solo su `Melampo-AI`.

Poi *Run all*. Alla fine si scarica `/kaggle/working/probe.json` e il log, da portare a Claude.

In [ ]:
# 0. Internet: senza, pip e il download del dataset falliscono ("Temporary failure in name resolution")
import socket
try:
    socket.gethostbyname("pypi.org")
    print("Internet OK")
except OSError:
    raise SystemExit("Internet spento: pannello a destra -> Session options -> Internet: On "
                     "(richiede il telefono verificato in Settings), poi Run All di nuovo.")

In [ ]:
!pip install -q voxtell TotalSegmentator nibabel
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# VoxTell segnala un problema di memoria con PyTorch 2.9.x: in quel caso si scende alla 2.8
import torch, subprocess, sys
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())
if torch.__version__.startswith("2.9"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.8.0", "torchvision==0.23.0"], check=True)
    print("installata torch 2.8.0: Run -> Restart & Run All")

In [ ]:
import os, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # repository pubblico: nessun token
url = f"https://{token}@github.com/francescoluciolattari/Melampo-AI.git" if token else "https://github.com/francescoluciolattari/Melampo-AI.git"
REPO = "/kaggle/working/melampo"
if os.path.isdir(os.path.join(REPO, ".git")):
    # già scaricato in un giro precedente: si aggiorna all'ultima versione del ramo principale
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", url], check=True)
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["rm", "-rf", REPO], check=True)   # una cartella lasciata a metà da un giro interrotto
    subprocess.run(["git", "clone", "--depth", "1", url, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Medical Segmentation Decathlon (CC BY-SA 4.0) sul registro AWS Open Data (bucket pubblico "msd-for-monai"),
# letto con boto3 senza credenziali, come indica il registro ("--no-sign-request").
# Si scaricano solo i casi che servono. Se l'elenco non trova Task06, la cella stampa cosa c'è nel bucket.
import os, glob, subprocess, sys, tarfile
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3"], check=True)
import boto3
from botocore import UNSIGNED
from botocore.config import Config

N = 5
DATA = "/kaggle/working/data"
os.makedirs(DATA, exist_ok=True)
images = sorted(glob.glob("/kaggle/input/**/imagesTr/lung_*.nii*", recursive=True))   # dati aggiunti con "Add Input"

for bucket, region in (("msd-for-monai", "us-west-2"), ("msd-for-monai-eu", "eu-west-2")):
    if images:
        break
    s3 = boto3.client("s3", region_name=region, config=Config(signature_version=UNSIGNED))
    try:
        keys = [o["Key"] for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket)
                for o in page.get("Contents", [])]
    except Exception as e:
        print(bucket, "elenco non disponibile:", repr(e)[:300]); continue
    print(bucket, len(keys), "oggetti; esempi:", keys[:15])
    lung = [k for k in keys if "lung" in k.lower()]
    print("con 'lung' nel nome:", lung[:20])
    imgs = sorted(k for k in lung if "imagesTr" in k and k.endswith(".nii.gz") and "/._" not in k)
    if imgs:
        for k in imgs[:N]:
            for kk in (k, k.replace("imagesTr", "labelsTr")):
                dest = os.path.join(DATA, kk)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                if not os.path.exists(dest):
                    s3.download_file(bucket, kk, dest)
    else:
        archive = next((k for k in lung if k.endswith((".tar", ".tar.gz", ".tgz"))), None)
        if archive:
            dest = os.path.join("/kaggle/working", os.path.basename(archive))
            print("scarico l'archivio", archive, "(circa 8,5 GB)")
            s3.download_file(bucket, archive, dest)
            with tarfile.open(dest) as t:
                members = [m for m in t.getmembers() if ("imagesTr/lung_" in m.name or "labelsTr/lung_" in m.name)
                           and "/._" not in m.name]
                wanted = sorted({os.path.basename(m.name) for m in members})[:N]
                t.extractall(DATA, members=[m for m in members if os.path.basename(m.name) in wanted])
            os.remove(dest)
    images = sorted(glob.glob(f"{DATA}/**/imagesTr/lung_*.nii.gz", recursive=True))
print(len(images), "immagini:", [os.path.basename(p) for p in images])
assert images, "nessun dato: mandare a Claude l'output di questa cella"

In [ ]:
import json, os
lobes = ["lung_upper_lobe_left", "lung_lower_lobe_left", "lung_upper_lobe_right", "lung_middle_lobe_right", "lung_lower_lobe_right"]
cases = [{"id": os.path.basename(p).split(".")[0], "image": p, "lesion": p.replace("imagesTr", "labelsTr"),
          "structure": lobes, "prompt": "tumore polmonare", "prompt_en": "lung tumor"} for p in images[:N]]
json.dump(cases, open("/kaggle/working/cases.json", "w"), indent=1)
print(len(cases), "casi")

In [ ]:
!cd /kaggle/working/melampo && python scripts/imaging_probe.py --cases /kaggle/working/cases.json --out /kaggle/working/probe.json --device cuda --fast-organs 2>&1 | tee /kaggle/working/probe.log

In [ ]:
import json
rows = json.load(open("/kaggle/working/probe.json"))
for r in rows:
    for k in ("prompt", "prompt_en"):
        if k in r:
            v = r[k]
            print(f"{r['id']:10} {k:9} {v['status']:10} Dice {v['dice']:.2f}  {v['seconds']:6.1f}s  "
                  f"misurato {v['measured_longest_axial_mm']} mm  riferimento {v['reference_longest_axial_mm']} mm")